In [ ]:
import os
from pathlib import Path
import pandas as pd

# ------- CONFIG -------
ROOT_DIR = r"C:\Users\israe\OneDrive\Documents\post doc\kfir\covid"
OUTPUT_CSV = "columns_value_counts_summary.csv"
INCLUDE_SHEET_COLUMN = True  # set to False if you don't want the 'sheet_name' column in the output
# ----------------------

EXCEL_EXTS = {".xls", ".xlsx", ".xlsm", ".xlsb"}
CSV_EXTS = {".csv"}

results = []

def count_non_empty(series: pd.Series) -> int:
    # Count cells that are not NaN and not just empty/whitespace
    not_na = series.notna()
    # Convert only non-na to string for speed; treat empty/whitespace as empty
    s = series[not_na].astype(str).str.strip()
    return (s != "").sum()

def process_csv(file_path: Path):
    # Try UTF-8 first, then a couple of common Windows encodings
    encodings_to_try = [None, "utf-8", "cp1255", "cp1252", "latin-1"]
    last_err = None
    for enc in encodings_to_try:
        try:
            col_counts = {}
            total_rows = 0
            for chunk in pd.read_csv(file_path, chunksize=100_000, encoding=enc, low_memory=False):
                total_rows += len(chunk)
                for col in chunk.columns:
                    col_counts.setdefault(col, 0)
                    col_counts[col] += count_non_empty(chunk[col])
            for col, nvals in col_counts.items():
                row = {
                    "path": str(file_path.parent),
                    "filename": file_path.name,
                    "column_name": col,
                    "num_values": int(nvals),
                }
                if INCLUDE_SHEET_COLUMN:
                    row["sheet_name"] = ""  # CSV has no sheets
                results.append(row)
            return
        except Exception as e:
            last_err = e
            continue
    # If all encodings failed, record an error row
    err_row = {
        "path": str(file_path.parent),
        "filename": file_path.name,
        "column_name": "<READ_ERROR>",
        "num_values": -1,
    }
    if INCLUDE_SHEET_COLUMN:
        err_row["sheet_name"] = ""
    results.append(err_row)

def process_excel(file_path: Path):
    # Try default engine first; fall back to pyxlsb for .xlsb if available
    engines_to_try = [None]
    if file_path.suffix.lower() == ".xlsb":
        engines_to_try = ["pyxlsb", None]
    last_err = None
    for eng in engines_to_try:
        try:
            # Read all sheets as dataframes
            sheets = pd.read_excel(file_path, sheet_name=None, engine=eng)
            for sheet_name, df in sheets.items():
                for col in df.columns:
                    nvals = count_non_empty(df[col])
                    row = {
                        "path": str(file_path.parent),
                        "filename": file_path.name,
                        "column_name": str(col),
                        "num_values": int(nvals),
                    }
                    if INCLUDE_SHEET_COLUMN:
                        row["sheet_name"] = str(sheet_name)
                    results.append(row)
            return
        except Exception as e:
            last_err = e
            continue
    # If all engines failed, record an error row
    err_row = {
        "path": str(file_path.parent),
        "filename": file_path.name,
        "column_name": "<READ_ERROR>",
        "num_values": -1,
    }
    if INCLUDE_SHEET_COLUMN:
        err_row["sheet_name"] = ""
    results.append(err_row)

def main():
    root = Path(ROOT_DIR)
    if not root.exists():
        raise FileNotFoundError(f"Directory not found: {root}")

    for dirpath, _, filenames in os.walk(root):
        for fname in filenames:
            fpath = Path(dirpath) / fname
            ext = fpath.suffix.lower()
            try:
                if ext in CSV_EXTS:
                    process_csv(fpath)
                elif ext in EXCEL_EXTS:
                    process_excel(fpath)
            except Exception as e:
                # Catch-all to ensure we still log problematic files
                err_row = {
                    "path": str(fpath.parent),
                    "filename": fpath.name,
                    "column_name": "<UNEXPECTED_ERROR>",
                    "num_values": -1,
                }
                if INCLUDE_SHEET_COLUMN:
                    err_row["sheet_name"] = ""
                results.append(err_row)

    # Create DataFrame and write summary
    cols = ["path", "filename"]
    if INCLUDE_SHEET_COLUMN:
        cols.append("sheet_name")
    cols += ["column_name", "num_values"]

    df = pd.DataFrame(results, columns=cols)
    df.sort_values(by=["path", "filename"] + (["sheet_name"] if INCLUDE_SHEET_COLUMN else []) + ["column_name"], inplace=True)
    df.to_csv(OUTPUT_CSV, index=False)
    print(f"Done. Summary written to: {OUTPUT_CSV}")
    print(df.head(20).to_string(index=False))

if __name__ == "__main__":
    main()
